# elec_voids_search: сравнение этапов поиска

Подготовлен без выполнения. Сырые кандидаты независимо из сетки и Вороного → общий список → кластеры близких точек с выбором максимального расстояния до ближайшего атомного центра → отсев по симметрии → геометрия. Все выбранные координаты неизменны; после геометрии дополнительных этапов нет.

Сохранена ограниченная выборка четырёх ранее неуспешных соединений. Чтобы подготовить расчёт всех семи структур, замените INPUTS на ALL_INPUTS. Ручные заряды заданы для всех структур. Визуализация отключена. Предыдущие численные результаты относятся к прежнему пайплайну.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from dataclasses import replace
from contextlib import redirect_stdout, redirect_stderr
from time import perf_counter
import ast
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "elec_voids_search/run_search.py").is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from elec_voids_search import DEFAULT_CONFIG, SearchConfig
from elec_voids_search.run_search import run_search

# Дефолты берутся из config.py. Любое поле можно изменить через replace.
SETTINGS = SearchConfig(probe_element="Li", probe_oxidation_state=1, export_cif=False)
# Примеры (раскомментируйте нужный):
# SETTINGS = replace(SETTINGS, grid_resolution_A=0.10)
# SETTINGS = replace(SETTINGS, merge_distance_A=1.0)  # более сильное объединение
# SETTINGS = replace(SETTINGS, merge_distance_A=0.60)  # более щадящее объединение
# SETTINGS = replace(SETTINGS, voronoi_image_range=2, symmetry_tolerance_A=0.05)
# SETTINGS = replace(SETTINGS, merge_distance_A=None)  # без общего объединения

# Другой ион (пример, не запускается автоматически):
# SETTINGS = replace(SETTINGS, probe_element="Na", probe_oxidation_state=1)
# SETTINGS = replace(SETTINGS, geometry_probe_spin="High Spin")
# Явные заряды относятся к каркасу; заряд внедряемого иона задан отдельно:
# SETTINGS = replace(SETTINGS, geometry_oxidation_states={"Li": 1, "Cl": -1})
# Последний запасной вариант (заряд-специфичный ionic_radius) разрешён по умолчанию.
# SETTINGS = replace(SETTINGS, geometry_probe_radius_override_A=0.76)  # явный пробник
# Допуск в % от локального радиуса пробника (дефолт 10%; 0 — строгая проверка):
# SETTINGS = replace(SETTINGS, geometry_overlap_tolerance_percent=10.0)

STAGES = ["grid", "voronoi", "combined", "merged", "nonequivalent",
          "geometry_evaluated", "geometry", "geometry_indeterminate",
          "geometry_rejected", "geometry_retained"]
VACANCY_TOLERANCE_A = 0.25  # критерий оценки известной позиции, не фильтр поиска
parameter_table = pd.DataFrame([
    {"parameter": key, "value": repr(value)} for key, value in SETTINGS.to_dict().items()
])
display(parameter_table)

run_id = datetime.now(ZoneInfo("Europe/Moscow")).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_DIR = PROJECT_ROOT / "elec_voids_results" / f"notebook_tests_{run_id}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir()
(OUTPUT_DIR / "settings.json").write_text(json.dumps(SETTINGS.to_dict(), indent=2))
print("Python:", sys.executable)
print("Результаты:", OUTPUT_DIR)

## Входные структуры

Все CIF из `kobernik_voids/cifs` и две структуры с искусственной Li-вакансией. Входные файлы не изменяются.

In [ ]:
TEST_FOLDER = PROJECT_ROOT / "kobernik_voids/cifs"
VACANCY_FILES = [
    PROJECT_ROOT / "cif_tests_2/LiCl_one_Li_vacancy.cif",
    PROJECT_ROOT / "cif_tests_2/Li3PO4_one_Li_vacancy.cif",
]
ALL_INPUTS = sorted(p for p in TEST_FOLDER.iterdir() if p.is_file() and p.suffix.lower() == ".cif") + VACANCY_FILES
MANUAL_OXIDATION_STATES = {
    "LATP_sc_integer152_stoichiometric": {
        "Li": 1,
        "Ti": 4,
        "Al": 3,
        "P": 5,
        "O": -2
    },
    "LLZO_cub_ga": {
        "Li": 1,
        "La": 3,
        "Zr": 4,
        "Ga": 3,
        "O": -2
    },
    "Li10Ge(PS6)2": {
        "Li": 1,
        "Ge": 4,
        "P": 5,
        "S": -2
    },
    "Li3PO4": {
        "Li": 1,
        "P": 5,
        "O": -2
    },
    "LiCl": {
        "Li": 1,
        "Cl": -1
    },
    "LiCl_one_Li_vacancy": {
        "Li": 1,
        "Cl": -1
    },
    "Li3PO4_one_Li_vacancy": {
        "Li": 1,
        "P": 5,
        "O": -2
    }
}
RETRY_NAMES = {"LLZO_cub_ga", "Li10Ge(PS6)2", "LiCl", "LiCl_one_Li_vacancy"}
INPUTS = [p for p in ALL_INPUTS if p.stem in RETRY_NAMES]
(OUTPUT_DIR / "manual_oxidation_states.json").write_text(json.dumps(MANUAL_OXIDATION_STATES, indent=2))
assert INPUTS and all(p.is_file() for p in INPUTS)
assert len({p.stem for p in INPUTS}) == len(INPUTS)
input_table = pd.DataFrame({
    "structure": [p.stem for p in INPUTS],
    "file": [str(p.relative_to(PROJECT_ROOT)) for p in INPUTS],
    "sha256": [hashlib.sha256(p.read_bytes()).hexdigest() for p in INPUTS],
})
input_table.to_csv(OUTPUT_DIR / "inputs.csv", index=False)
display(input_table[["structure", "file"]])


## Источники и отсутствие консольных аргументов

Вычислительные блоки взяты из `test_easy.ipynb` и `single_structure_voronoi.ipynb`. Для доступности всех настроек параметризованы сигнатуры и значения внутри вызовов. Поэтому целиком сравниваются только функции, которые остались дословными копиями; остальные изменения описаны в файлах происхождения и PARAMETERS.md. Исходные notebook и исходный `void_search.py` не перезаписываются.

In [ ]:
package_dir = PROJECT_ROOT / "elec_voids_search"
original_paths = [PROJECT_ROOT / "kobernik_voids/src/test_easy.ipynb",
                  PROJECT_ROOT / "kobernik_voids/src/void_search.py",
                  PROJECT_ROOT / "single_structure_voronoi.ipynb"]
original_hashes = {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in original_paths}
copied = json.loads((package_dir / "COPIED_FUNCTIONS.json").read_text())
for entry in copied:
    hashes = []
    for key in ["source", "destination"]:
        text = (PROJECT_ROOT / entry[key]).read_text()
        node = next(n for n in ast.parse(text).body if isinstance(n, ast.FunctionDef) and n.name == entry["function"])
        block = "".join(text.splitlines(keepends=True)[node.lineno-1:node.end_lineno]).rstrip(chr(10))
        hashes.append(hashlib.sha256(block.encode()).hexdigest())
    assert hashes[0] == hashes[1] == entry["function_source_sha256"], entry["function"]
main_source = (package_dir / "run_search.py").read_text()
assert not any(token in main_source for token in ["argparse", "parse_args", "sys.argv"])
print("Дословные функции соответствуют источникам; все остальные параметры заданы через SearchConfig.")
geometry_sources = json.loads((package_dir / "GEOMETRY_SOURCE.json").read_text())
for entry in geometry_sources["module_copies"]:
    source_bytes = (PROJECT_ROOT / entry["source"]).read_bytes()
    destination_bytes = (PROJECT_ROOT / entry["destination"]).read_bytes()
    assert source_bytes == destination_bytes
    assert hashlib.sha256(source_bytes).hexdigest() == entry["sha256"]
print("shannon_radii.py и site_atomic_data.py совпадают с оригиналами побайтно.")


## Запуск выбранных структур и сохранение этапов

`geometry_evaluated` содержит оценки ровно всех представителей из `nonequivalent`, с теми же ID и координатами. `geometry` — окончательная прошедшая выборка; `geometry_rejected` — отклонённые точки. `geometry_indeterminate` — пустая таблица совместимости, `geometry_retained` совпадает с `geometry`.

`geometry_candidate_audit` сохраняет радиусные оценки проверенных точек. Оптимизации, перемещения, возврата отброшенных кандидатов и повторного объединения нет. Ошибка одной структуры записывается в failures.csv; остальные продолжают рассчитываться.


In [ ]:
results, summaries, failures = {}, [], []
for number, path in enumerate(INPUTS, 1):
    print(f"[{number}/{len(INPUTS)}] {path.name}", flush=True)
    started = perf_counter()
    try:
        with (LOG_DIR / f"{path.stem}.log").open("w", buffering=1) as log, redirect_stdout(log), redirect_stderr(log):
            structure_settings = replace(SETTINGS, geometry_oxidation_states=MANUAL_OXIDATION_STATES[path.stem])
            result = run_search(path, OUTPUT_DIR / path.stem, config=structure_settings)
    except Exception as exc:
        failures.append({"structure": path.stem, "error": repr(exc)})
        pd.DataFrame(failures).to_csv(OUTPUT_DIR / "failures.csv", index=False)
        print("Ошибка:", repr(exc), flush=True)
        continue
    results[path.stem] = result
    summary = result["summary"].copy()
    summary["elapsed_seconds"] = perf_counter() - started
    summaries.append(summary)
    pd.concat(summaries, ignore_index=True).to_csv(OUTPUT_DIR / "all_structures_summary.csv", index=False)
    print("После симметрии:", len(result["nonequivalent"]),
          "; геометрия:", len(result["geometry"]),
          "; неопределённо:", len(result["geometry_indeterminate"]), flush=True)
all_summary = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
display(all_summary)
display(pd.DataFrame(failures, columns=["structure", "error"]))


## Проверки порядка этапов, координат и происхождения

Проверяются сохранение обеих исходных выборок до группировки, полное покрытие ID кластерами, принадлежность каждого представителя исходному списку и запись действующих настроек. Координаты до геометрии не должны усредняться. Представитель выбирается по максимальному расстоянию до ближайшего атомного центра. Дополнительная колонка показывает размах кластера относительно его представителя.
Проверки требуют неизменности дробных и декартовых координат на всех этапах и невозрастания численности combined ≥ merged ≥ nonequivalent ≥ geometry.


In [ ]:
table_checks = []
for name, result in results.items():
    grid, voronoi = result["grid"], result["voronoi"]
    combined, merged = result["combined"], result["merged"]
    selected, clusters = result["nonequivalent"], result["clusters"]
    assert len(combined) == len(grid) + len(voronoi)
    assert len(voronoi) == result["summary"].iloc[0].n_voronoi_raw_vertices
    assert combined.candidate_id.is_unique
    assert set(merged.candidate_id) == set(combined.loc[combined.is_merge_representative, "candidate_id"])
    assert set(selected.candidate_id) <= set(merged.candidate_id)
    assert set(selected.candidate_id) == set(merged.loc[merged.is_symmetry_representative, "candidate_id"])
    all_members = [candidate for members in clusters.member_candidate_ids for candidate in members]
    assert sorted(all_members) == sorted(combined.candidate_id.tolist())
    assert len(all_members) == len(combined)
    for method, original in [("grid", grid), ("voronoi", voronoi)]:
        part = combined[combined.source_method == method]
        assert part.source_candidate_id.tolist() == original.candidate_id.tolist()
        np.testing.assert_allclose(part[["frac_x", "frac_y", "frac_z"]], original[["frac_x", "frac_y", "frac_z"]], atol=0, rtol=0)
    if SETTINGS.merge_grouping == "representative" and SETTINGS.merge_distance_A is not None:
        assert (clusters.max_distance_to_representative_A < SETTINGS.merge_distance_A).all()
    if SETTINGS.merge_representative == "largest_center_distance":
        for _, cluster in clusters.iterrows():
            group = combined[combined.merge_cluster_id == cluster.merge_cluster_id]
            chosen = group[group.candidate_id == cluster.representative_candidate_id].iloc[0]
            assert chosen.nearest_atom_center_A == group.nearest_atom_center_A.max()
    original_rows = combined.set_index("candidate_id")
    np.testing.assert_allclose(merged[["frac_x", "frac_y", "frac_z"]],
                               original_rows.loc[merged.candidate_id, ["frac_x", "frac_y", "frac_z"]], atol=0, rtol=0)
    for stage in STAGES:
        assert len(pd.read_csv(OUTPUT_DIR / name / f"{stage}_candidates.csv")) == len(result[stage])
    evaluated = result["geometry_evaluated"]
    assert len(evaluated) == len(selected)
    assert len(result["geometry"]) + len(result["geometry_rejected"]) + len(result["geometry_indeterminate"]) == len(selected)
    assert set(evaluated.candidate_id) == set(selected.candidate_id)
    assert set(result["geometry_candidate_audit"].candidate_id) == set(selected.candidate_id)
    assert (result["geometry"].geometry_status == "passed").all()
    if len(evaluated):
        host = Structure.from_file(next(p for p in INPUTS if p.stem == name))
        np.testing.assert_allclose(evaluated[["cart_x_A", "cart_y_A", "cart_z_A"]],
                                   evaluated[["frac_x", "frac_y", "frac_z"]].to_numpy() @ host.lattice.matrix,
                                   atol=1e-9, rtol=0)
    selection_stages = ["combined", "merged", "nonequivalent", "geometry"]
    counts = [len(result[stage]) for stage in selection_stages]
    assert all(a >= b for a, b in zip(counts, counts[1:]))
    coordinates = ["frac_x", "frac_y", "frac_z", "cart_x_A", "cart_y_A", "cart_z_A"]
    for stage in ["merged", "nonequivalent", "geometry_evaluated", "geometry"]:
        table = result[stage]
        assert np.array_equal(original_rows.loc[table.candidate_id, coordinates].to_numpy(),
                              table[coordinates].to_numpy())
    metadata = json.loads((OUTPUT_DIR / name / "metadata.json").read_text())
    expected_settings = replace(SETTINGS, geometry_oxidation_states=MANUAL_OXIDATION_STATES[name]).to_dict()
    assert metadata["parameters"]["settings"] == expected_settings == result["settings"]
    assert metadata["parameters"]["symmetry_reduction_stage"] == "before_geometry_only"
    assert not len(result["geometry_indeterminate"])
    assert np.isfinite(result["site_radii"].radius_A).all()
    assert np.isfinite(result["geometry_evaluated"].probe_radius_A).all()
    table_checks.append({"structure": name, "passed": True,
                         "max_cluster_span_A": float(clusters.max_distance_to_representative_A.max()) if len(clusters) else 0})
display(pd.DataFrame(table_checks))

## Контроль известных вакансий

Эталон определяется из исходной и дефектной структуры. Порог 0,25 Å оценивает восстановление известного Li; он не участвует в группировке или поиске. Новые значения допусков не считаются валидированными до выполнения этих проверок.

In [ ]:
def get_removed_li_position(pristine, defect):
    np.testing.assert_allclose(pristine.lattice.matrix, defect.lattice.matrix, atol=1e-7)
    assert len(pristine) == len(defect) + 1
    distance = pristine.lattice.get_all_distances(pristine.frac_coords, defect.frac_coords)
    missing, owners = [], []
    for i, site in enumerate(pristine):
        same_element = np.array([other.specie.symbol == site.specie.symbol for other in defect])
        matches = np.flatnonzero(same_element & (distance[i] < 1e-6))
        if len(matches):
            assert len(matches) == 1
            owners.append(int(matches[0]))
        else:
            missing.append(i)
    assert len(missing) == 1 and pristine[missing[0]].specie.symbol == "Li"
    assert sorted(owners) == list(range(len(defect)))
    return pristine[missing[0]].frac_coords

vacancy_rows = []
for material in (["LiCl", "Li3PO4"] if SETTINGS.probe_element == "Li" and SETTINGS.probe_oxidation_state == 1 else []):
    pristine = Structure.from_file(PROJECT_ROOT / f"cif_tests_2/{material}.cif")
    name = f"{material}_one_Li_vacancy"
    defect = Structure.from_file(PROJECT_ROOT / f"cif_tests_2/{name}.cif")
    if name not in results:
        continue
    target = get_removed_li_position(pristine, defect)
    operations = SpacegroupAnalyzer(defect, symprec=SETTINGS.symmetry_symprec_A,
                                   angle_tolerance=SETTINGS.symmetry_angle_tolerance_deg).get_symmetry_operations()
    target_orbit = np.array([operation.operate(target) % 1 for operation in operations])
    for stage in STAGES:
        table = results[name][stage]
        nearest = float(defect.lattice.get_all_distances([target], table[["frac_x", "frac_y", "frac_z"]]).min()) if len(table) else float("inf")
        orbit_nearest = float(defect.lattice.get_all_distances(target_orbit, table[["frac_x", "frac_y", "frac_z"]]).min()) if len(table) else float("inf")
        vacancy_rows.append({"structure": name, "stage": stage,
                             "nearest_vacancy_A": nearest, "nearest_symmetry_orbit_A": orbit_nearest,
                             "recovered": orbit_nearest < VACANCY_TOLERANCE_A})
vacancy_recovery = pd.DataFrame(vacancy_rows, columns=["structure", "stage", "nearest_vacancy_A", "nearest_symmetry_orbit_A", "recovered"])
vacancy_recovery.to_csv(OUTPUT_DIR / "vacancy_recovery.csv", index=False)
display(vacancy_recovery)
# Не прерываем статистику при потере вакансии: это результат сравнения.
# Отброшенные/неопределённые таблицы диагностические, не этапы успешного отбора.
recovery_stages = ["grid", "voronoi", "combined", "merged", "nonequivalent", "geometry", "geometry_retained"]
display(vacancy_recovery[vacancy_recovery.stage.isin(recovery_stages)].pivot(index="structure", columns="stage", values="recovered"))

## Общие координаты и проверка полного размера иона

Следующие таблицы объединяют все успешно рассчитанные структуры. Для каждого этапа сохраняются координаты, исходные ID и результаты геометрии. `full_radius_status` всегда проверяет полный найденный радиус пробника. Основной критерий использует geometry_minimum_fit_fraction (дефолт 1.0); при дефолте критерии совпадают. При явном включении fallback источник радиуса записывается отдельно.

Контроли занятых сайтов выбранного пробного элемента используют удаление центрального сайта из окружения, без оптимизации. Берётся радиус самого занятого сайта с его исходным окружением; проверка не является независимым поиском неизвестного междоузлия. Координаты искусственных вакансий не передаются в поиск. Контроли сохраняются только для выбранного элемента и заряда.


In [ ]:
coordinate_tables, counts, model_rows, control_tables, radius_tables = [], [], [], [], []
for name, result in results.items():
    for stage in STAGES:
        table = result[stage].copy()
        table.insert(0, "structure", name)
        table.insert(1, "stage", stage)
        coordinate_tables.append(table)
        counts.append({"structure": name, "stage": stage, "n_candidates": len(table)})
    evaluated = result["geometry_evaluated"]
    for primary in ["passed", "indeterminate", "rejected"]:
        for strict in ["passed", "indeterminate", "rejected"]:
            model_rows.append({"structure": name, "geometry_status": primary,
                               "full_radius_status": strict,
                               "count": int(((evaluated.geometry_status == primary) &
                                             (evaluated.full_radius_status == strict)).sum())})
    for key, target in [("occupied_probe_controls", control_tables), ("site_radii", radius_tables)]:
        table = result[key].copy()
        table.insert(0, "structure", name)
        target.append(table)
counts = pd.DataFrame(counts, columns=["structure", "stage", "n_candidates"])
counts.to_csv(OUTPUT_DIR / "stage_counts.csv", index=False)
if len(counts):
    display(counts.pivot(index="structure", columns="stage", values="n_candidates").reindex(columns=STAGES))
for filename, tables in [("all_stage_coordinates", coordinate_tables),
                         ("all_occupied_probe_controls", control_tables), ("all_site_radii", radius_tables)]:
    table = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
    table.to_csv(OUTPUT_DIR / f"{filename}.csv", index=False)
    (OUTPUT_DIR / f"{filename}.json").write_text(table.to_json(orient="records", indent=2, double_precision=15))
    if filename != "all_stage_coordinates":
        display(table)
model_comparison = pd.DataFrame(model_rows, columns=["structure", "geometry_status", "full_radius_status", "count"])
model_comparison.to_csv(OUTPUT_DIR / "geometry_fit_fraction_comparison.csv", index=False)
display(model_comparison[model_comparison["count"] > 0])
# Итоговые координаты (полные данные сохранены в CSV/JSON):
for name, result in results.items():
    print(name)
    display(result["geometry"][["candidate_id", "frac_x", "frac_y", "frac_z",
                                "probe_element", "probe_oxidation_state", "probe_coordination",
                                "probe_radius_A", "probe_radius_source", "free_radius_A",
                                "geometry_margin_A", "full_radius_margin_A"]])


In [ ]:
for path, expected_hash in original_hashes.items():
    assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == expected_hash
validation = {"structures_tested": len(results), "failures": failures, "table_checks": table_checks,
              "vacancy_checks": vacancy_recovery.to_dict("records"),
              "original_files_unchanged": True, "common_settings": SETTINGS.to_dict(),
              "manual_oxidation_states": MANUAL_OXIDATION_STATES,
              "stage_order": ["grid_and_voronoi", "combined", "merged", "nonequivalent", "geometry"]}
(OUTPUT_DIR / "validation.json").write_text(json.dumps(validation, ensure_ascii=False, indent=2))
print("Структурные проверки таблиц завершены. Восстановление вакансий оцените по vacancy_recovery.csv. Результаты:", OUTPUT_DIR)